# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

I am using a **RandomForestRegressor** to predict `ctr` using the `signal_features` (content-specific attributes like word count and intent) rather than outcome metrics.

**Why it fits:** This lane focuses on a "what drives X" question. A Random Forest is robust to different feature scales and types, and unlike a black-box classifier, it allows for **permutation importance** analysis. This lets us measure which signals actually impact performance, providing actionable decision-support rather than just a prediction.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.inspection import permutation_importance

# 1. Method choice and why
# Method: RandomForestRegressor
# Reason: Fits the 'what drives X' requirement. It handles non-linear relationships
# and allows for permutation importance analysis to identify key drivers of CTR.

# Load data using relative path pattern
df = pd.read_csv('my-ml-starter/data/raw/content_refresh_anonymized.csv')

# Define features (Signal only, no outcomes)
signal_features = [
    'word_count', 'char_count', 'content_age_days', 'days_since_last_update',
    'content_type', 'main_intent', 'competition_level', 'search_volume',
    'competition', 'cpc', 'word_count_tier', 'age_tier', 'freshness_tier'
]
target = 'ctr'

print(f"Data loaded: {df.shape[0]} rows.")

Data loaded: 30000 rows.


## 2. Split design

With only 32 clients and ~937 rows each, a plain random split would likely leak client-specific styles or industry niches into both sets, resulting in over-optimistic results.

I used `GroupShuffleSplit` (test_size=0.2) keyed on `client_id`. This ensures the model is tested on entirely unseen clients, which is an "honest" test of whether content signals generalize across different domains or if CTR is primarily driven by the specific client brand.

In [ ]:
# 2. Split design
# Checking client distribution
client_counts = df['client_id'].value_counts()
print(f"Unique clients: {df['client_id'].nunique()}")
print("Client row distribution:")
print(client_counts.describe())

# Grouped split to prevent leakage
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df['client_id']))

train_df = df.iloc[train_idx]
test_df = df.iloc[test_idx]

# Assert no overlap
train_clients = set(train_df['client_id'])
test_clients = set(test_df['client_id'])
overlap = train_clients & test_clients
print(f"Overlap set: {overlap}")
assert len(overlap) == 0

Unique clients: 32
Client row distribution:
count      32.000000
mean      937.500000
std      1376.387113
min         3.000000
25%       110.250000
50%       567.000000
75%      1058.750000
max      7008.000000
Name: count, dtype: float64
Overlap set: set()


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [ ]:
# Section 3: Train + compare vs baseline

def preprocess_signals(data, features, is_train=True, train_cols=None):
    X = data[features].copy()

    # Fill missing values: numeric with -1, categorical with 'unknown'
    numeric_cols = X.select_dtypes(include=[np.number]).columns
    categorical_cols = X.select_dtypes(exclude=[np.number]).columns

    X[numeric_cols] = X[numeric_cols].fillna(-1)
    X[categorical_cols] = X[categorical_cols].fillna('unknown')

    # One-hot encoding
    X = pd.get_dummies(X, columns=categorical_cols)

    if not is_train and train_cols is not None:
        # Ensure test set has same columns as train set
        for col in set(train_cols) - set(X.columns):
            X[col] = 0
        X = X[train_cols]

    return X

# Prepare features
X_train = preprocess_signals(train_df, signal_features)
train_cols = X_train.columns.tolist()
X_test = preprocess_signals(test_df, signal_features, is_train=False, train_cols=train_cols)

y_train = train_df[target]
y_test = test_df[target]

# Baseline: Predict Mean CTR
baseline_preds = np.full_like(y_test, y_train.mean())
baseline_mae = mean_absolute_error(y_test, baseline_preds)
baseline_r2 = r2_score(y_test, baseline_preds)

# Model: Random Forest
rf = RandomForestRegressor(n_estimators=200, max_depth=8, random_state=42)
rf.fit(X_train, y_train)
model_preds = rf.predict(X_test)
model_mae = mean_absolute_error(y_test, model_preds)
model_r2 = r2_score(y_test, model_preds)

# Comparison Table
results = pd.DataFrame({
    'MAE': [baseline_mae, model_mae],
    'R2': [baseline_r2, model_r2]
}, index=['Baseline (predict mean CTR)', 'Model (RandomForestRegressor)'])

print("Model Comparison (on Held-out Grouped Test Split):")
display(results)

Model Comparison (on Held-out Grouped Test Split):


,MAE,R2
Baseline (predict mean CTR),0.553653,-0.065705
Model (RandomForestRegressor),0.873754,-1.885138


## 4. Errors and interpretation

### Error Analysis
For the top errors above:
1. The largest error often occurs on high-outlier CTR rows where content signals (like word count) look standard, but the actual performance was exceptional.
2. In cases where the model under-predicts, the `content_type` may have unique viral appeal not captured by raw word counts.
3. Large errors also appear for niche `search_volume` keywords where `ctr` is highly volatile due to small sample sizes.

### Honest Performance Assessment
Based on the table in Section 3, the model's R2 is **-1.89 (compared to the baseline's -0.07)**. Because R2 is negative and substantially worse than the baseline, the model actively performs worse than guessing the mean CTR for every row. This suggests that CTR in this dataset is highly dependent on specific client brand authority or external factors not captured in the `signal_features` (e.g., visual layout, brand recognition). The content signals alone provide only directional support rather than high-precision predictive power for unseen clients.

In [ ]:
# Section 4: Errors and interpretation

# 1. Permutation Importance
perm_importance = permutation_importance(rf, X_test, y_test, n_repeats=10, random_state=42)
importance_df = pd.DataFrame({'feature': X_test.columns, 'importance': perm_importance.importances_mean})
print("Top 8 Features by Permutation Importance:")
display(importance_df.sort_values(by='importance', ascending=False).head(8))

# 2. Error Analysis: Largest Absolute Errors
error_df = test_df[['content_id', 'content_type', 'word_count', 'search_volume', 'ctr']].copy()
error_df['predicted_ctr'] = model_preds
error_df['abs_error'] = (error_df['ctr'] - error_df['predicted_ctr']).abs()

top_errors = error_df.sort_values(by='abs_error', ascending=False).head(3)
print("\nTop 3 Errors (Largest Absolute Error):")
display(top_errors)

# Interpretation notes for user/markdown follows in next cell

Top 8 Features by Permutation Importance:


,feature,importance
0,word_count,2.807432
1,char_count,2.628358
2,content_age_days,1.036500
4,search_volume,0.525834
24,age_tier_181-365,0.062667
27,age_tier_91-180,0.057420
5,competition,0.025829
11,main_intent_informational,0.001828



Top 3 Errors (Largest Absolute Error):


,content_id,content_type,word_count,search_volume,ctr,predicted_ctr,abs_error
4500,content_f2eeaeefe9f1,keyword article,3588.0,10.0,33.33,0.228433,33.101567
22847,content_77c30aa8260f,keyword article,1987.0,0.0,33.33,0.380013,32.949987
9129,content_3db3c9053d5c,keyword article,3462.0,0.0,25.00,0.325801,24.674199


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.